# Metagenome Diversity Analysis — automated pipeline

Project layout:

```text
project/
├── pipeline.ipynb
├── data/
└── results/
```

The notebook keeps the TP commands and parameters unchanged.  
Each command cell deletes only its own previous output, reruns the command, prints terminal-style output, and saves the full output in `results/logs/`.


## 0. Setup

In [1]:

from pathlib import Path
import subprocess, shutil

ROOT = Path.cwd()
DATA = ROOT / "data" / "TP_Diversity_data"
RESULTS = ROOT / "results"
WORK = RESULTS / "work"
LOGS = RESULTS / "logs"

WORK.mkdir(parents=True, exist_ok=True)
LOGS.mkdir(parents=True, exist_ok=True)

inputs = [
    "reads.fastq",
    "groups.txt",
    "sortmerna_ssu_db.fasta",
    "silva_ssu_db.fasta",
    "silva_ssu_db_taxa.txt",
    "helper.py",
]

missing = [x for x in inputs if not (DATA / x).exists()]
if missing:
    raise FileNotFoundError("Missing in data/: " + ", ".join(missing))

for name in inputs:
    dst = WORK / name
    if not dst.exists():
        dst.symlink_to((DATA / name).resolve())

print("WORK:", WORK)


WORK: /home/gman/Documents/MS-GENIOMHE/M2-GENIOMHE/3. Advanced Genomics/3. Metagenomics/automated/results/work


## Helpers

In [2]:

def remove(*names):
    for name in names:
        p = WORK / name
        if p.is_dir():
            shutil.rmtree(p)
        elif p.exists() or p.is_symlink():
            p.unlink()

def remove_glob(pattern):
    for p in WORK.glob(pattern):
        if p.is_dir():
            shutil.rmtree(p)
        elif p.exists() or p.is_symlink():
            p.unlink()

def run(command, log_name):
    print("$", command)
    p = subprocess.Popen(
        command,
        cwd=WORK,
        shell=True,
        executable="/bin/bash",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    out = []
    for line in p.stdout:
        print(line, end="")
        out.append(line)
    code = p.wait()
    (LOGS / log_name).write_text("".join(out), encoding="utf-8")
    if code != 0:
        raise RuntimeError(f"Command failed. See results/logs/{log_name}")

def run_mothur(command, log_name):
    run(f'printf "{command}\\nquit()\\n" | mothur', log_name)


# I. Extraction of 16S/18S rDNA reads

```bash
sortmerna --ref sortmerna_ssu_db.fasta --reads reads.fastq --workdir sortmerna_out --aligned ./aligned --fastx -e 0.001
```

In [3]:

remove("sortmerna_out", "aligned.fq", "aligned.log")
run(
    "sortmerna --ref sortmerna_ssu_db.fasta --reads reads.fastq "
    "--workdir sortmerna_out --aligned ./aligned --fastx -e 0.001",
    "01_sortmerna.log"
)


$ sortmerna --ref sortmerna_ssu_db.fasta --reads reads.fastq --workdir sortmerna_out --aligned ./aligned --fastx -e 0.001
[process:1393] === Options processing starts ... ===
Found value: sortmerna
Found flag: --ref
Found value: sortmerna_ssu_db.fasta of previous flag: --ref
Found flag: --reads
Found value: reads.fastq of previous flag: --reads
Found flag: --workdir
Found value: sortmerna_out of previous flag: --workdir
Found flag: --aligned
Found value: ./aligned of previous flag: --aligned
Found flag: --fastx
Previous flag: --fastx is Boolean. Setting to True
Found flag: -e
Found value: 0.001 of previous flag: -e
[opt_workdir:995] Using WORKDIR: "/home/gman/Documents/MS-GENIOMHE/M2-GENIOMHE/3. Advanced Genomics/3. Metagenomics/automated/results/work/sortmerna_out" as specified
[process:1483] Processing option: aligned with value: ./aligned
[process:1483] Processing option: e with value: 0.001
[process:1483] Processing option: fastx with value: 
[process:1483] Processing option: reads

### Question 1

In [4]:

total = sum(1 for _ in open(WORK / "reads.fastq")) // 4
retained = sum(1 for _ in open(WORK / "aligned.fq")) // 4
print("Total:", total)
print("Retained:", retained)
print("Removed:", total - retained)


Total: 100000
Retained: 98093
Removed: 1907


# II. Quality control

## FastQC before filtering
```bash
fastqc aligned.fq
```

In [5]:

remove("aligned_fastqc.html", "aligned_fastqc.zip")
run("fastqc aligned.fq", "02_fastqc_before.log")
print("Report:", WORK / "aligned_fastqc.html")


$ fastqc aligned.fq
null
Started analysis of aligned.fq
Approx 5% complete for aligned.fq
Approx 10% complete for aligned.fq
Approx 15% complete for aligned.fq
Approx 20% complete for aligned.fq
Approx 25% complete for aligned.fq
Approx 30% complete for aligned.fq
Too many tiles (>2500) so giving up trying to do per-tile qualities since we're probably parsing the file wrongly
Approx 35% complete for aligned.fq
Approx 40% complete for aligned.fq
Approx 45% complete for aligned.fq
Approx 50% complete for aligned.fq
Approx 55% complete for aligned.fq
Approx 60% complete for aligned.fq
Approx 65% complete for aligned.fq
Approx 70% complete for aligned.fq
Approx 75% complete for aligned.fq
Approx 80% complete for aligned.fq
Approx 85% complete for aligned.fq
Approx 90% complete for aligned.fq
Approx 95% complete for aligned.fq
Analysis complete for aligned.fq
Report: /home/gman/Documents/MS-GENIOMHE/M2-GENIOMHE/3. Advanced Genomics/3. Metagenomics/automated/results/work/aligned_fastqc.html


## Sickle filtering
```bash
sickle se -f aligned.fq -t sanger -o trimmed.fq -q 30 -l 30
```

In [6]:

remove("trimmed.fq")
run(
    "sickle se -f aligned.fq -t sanger -o trimmed.fq -q 30 -l 30",
    "03_sickle.log"
)


$ sickle se -f aligned.fq -t sanger -o trimmed.fq -q 30 -l 30

SE input file: aligned.fq

Total FastQ records: 98093
FastQ records kept: 90779
FastQ records discarded: 7314



### Question 3

In [7]:

before = sum(1 for _ in open(WORK / "aligned.fq")) // 4
after = sum(1 for _ in open(WORK / "trimmed.fq")) // 4
print("Before:", before)
print("After:", after)
print("Removed:", before - after)


Before: 98093
After: 90779
Removed: 7314


## FastQC after filtering
```bash
fastqc trimmed.fq
```

In [8]:

remove("trimmed_fastqc.html", "trimmed_fastqc.zip")
run("fastqc trimmed.fq", "04_fastqc_after.log")
print("Report:", WORK / "trimmed_fastqc.html")


$ fastqc trimmed.fq
null
Started analysis of trimmed.fq
Approx 5% complete for trimmed.fq
Approx 10% complete for trimmed.fq
Approx 15% complete for trimmed.fq
Approx 20% complete for trimmed.fq
Too many tiles (>2500) so giving up trying to do per-tile qualities since we're probably parsing the file wrongly
Approx 25% complete for trimmed.fq
Approx 30% complete for trimmed.fq
Approx 35% complete for trimmed.fq
Approx 40% complete for trimmed.fq
Approx 45% complete for trimmed.fq
Approx 50% complete for trimmed.fq
Approx 55% complete for trimmed.fq
Approx 60% complete for trimmed.fq
Approx 65% complete for trimmed.fq
Approx 70% complete for trimmed.fq
Approx 75% complete for trimmed.fq
Approx 80% complete for trimmed.fq
Approx 85% complete for trimmed.fq
Approx 90% complete for trimmed.fq
Approx 95% complete for trimmed.fq
Analysis complete for trimmed.fq
Report: /home/gman/Documents/MS-GENIOMHE/M2-GENIOMHE/3. Advanced Genomics/3. Metagenomics/automated/results/work/trimmed_fastqc.html


## FASTQ → FASTA
```bash
seqtk seq -A trimmed.fq > trimmed.fa
```

In [9]:

remove("trimmed.fa")
run("seqtk seq -A trimmed.fq > trimmed.fa", "05_seqtk.log")


$ seqtk seq -A trimmed.fq > trimmed.fa


# III. OTUs and diversity analyses

## OTU clustering
```bash
mothur
>cluster(fasta=trimmed.fa, method=agc, cutoff=0.03)
```

In [10]:

remove_glob("trimmed.agc*")
remove_glob("trimmed.unique.fa*")
run_mothur(
    "cluster(fasta=trimmed.fa, method=agc, cutoff=0.03)",
    "06_mothur_cluster.log"
)


$ printf "cluster(fasta=trimmed.fa, method=agc, cutoff=0.03)\nquit()\n" | mothur
Linux version

Using ReadLine
mothur v.1.48.0
Last updated: 5/20/22
by
Patrick D. Schloss

Department of Microbiology & Immunology

University of Michigan
http://www.mothur.org

When using, please cite:
Schloss, P.D., et al., Introducing mothur: Open-source, platform-independent, community-supported software for describing and comparing microbial communities. Appl Environ Microbiol, 2009. 75(23):7537-41.

Distributed under the GNU General Public License

Type 'help()' for information on the commands that are available

For questions and analysis support, please visit our forum at https://forum.mothur.org

Type 'quit()' to exit program

[NOTE]: Setting random seed to 19760620.

Interactive Mode



mothur > cluster(fasta=trimmed.fa, method=agc, cutoff=0.03)
Unable to open vsearch. Trying mothur's executable directory vsearch.
Unable to open vsearch.
vsearch file does not exist. Checking path... 
Found vsearc

## List retained sequences
```bash
mothur
>list.seqs(list=trimmed.agc.list)
```

In [11]:

remove("trimmed.agc.accnos")
run_mothur(
    "list.seqs(list=trimmed.agc.list)",
    "07_mothur_list_seqs.log"
)


$ printf "list.seqs(list=trimmed.agc.list)\nquit()\n" | mothur
Linux version

Using ReadLine
mothur v.1.48.0
Last updated: 5/20/22
by
Patrick D. Schloss

Department of Microbiology & Immunology

University of Michigan
http://www.mothur.org

When using, please cite:
Schloss, P.D., et al., Introducing mothur: Open-source, platform-independent, community-supported software for describing and comparing microbial communities. Appl Environ Microbiol, 2009. 75(23):7537-41.

Distributed under the GNU General Public License

Type 'help()' for information on the commands that are available

For questions and analysis support, please visit our forum at https://forum.mothur.org

Type 'quit()' to exit program

[NOTE]: Setting random seed to 19760620.

Interactive Mode



mothur > list.seqs(list=trimmed.agc.list)

Output File Names: 
trimmed.agc.accnos


mothur > quit()
Logfile : mothur.1790927827.logfile



## Filter groups
```bash
python3 helper.py group-subset trimmed.agc.accnos groups.txt
```

In [12]:

remove("groups_filtered.txt")
run(
    "python3 helper.py group-subset trimmed.agc.accnos groups.txt",
    "08_group_subset.log"
)


$ python3 helper.py group-subset trimmed.agc.accnos groups.txt
Filtered group file written at groups_filtered.txt


## Make shared file
```bash
mothur
>make.shared(list=trimmed.agc.list, group=groups_filtered.txt)
```

In [14]:

remove("trimmed.agc.shared")
run_mothur(
    "make.shared(list=trimmed.agc.list, group=groups_filtered.txt)",
    "09_mothur_make_shared.log"
)


$ printf "make.shared(list=trimmed.agc.list, group=groups_filtered.txt)\nquit()\n" | mothur
Linux version

Using ReadLine
mothur v.1.48.0
Last updated: 5/20/22
by
Patrick D. Schloss

Department of Microbiology & Immunology

University of Michigan
http://www.mothur.org

When using, please cite:
Schloss, P.D., et al., Introducing mothur: Open-source, platform-independent, community-supported software for describing and comparing microbial communities. Appl Environ Microbiol, 2009. 75(23):7537-41.

Distributed under the GNU General Public License

Type 'help()' for information on the commands that are available

For questions and analysis support, please visit our forum at https://forum.mothur.org

Type 'quit()' to exit program

[NOTE]: Setting random seed to 19760620.

Interactive Mode



mothur > make.shared(list=trimmed.agc.list, group=groups_filtered.txt)
0.03

Output File Names:
trimmed.agc.shared


mothur > quit()
Logfile : mothur.1790927869.logfile



## Alpha-diversity
```bash
mothur
>summary.single(shared=trimmed.agc.shared, groupmode=t, calc=sobs-shannon-chao-invsimpson)
```

In [15]:

remove("trimmed.agc.groups.summary")
run_mothur(
    "summary.single(shared=trimmed.agc.shared, groupmode=t, calc=sobs-shannon-chao-invsimpson)",
    "10_mothur_alpha.log"
)


$ printf "summary.single(shared=trimmed.agc.shared, groupmode=t, calc=sobs-shannon-chao-invsimpson)\nquit()\n" | mothur
Linux version

Using ReadLine
mothur v.1.48.0
Last updated: 5/20/22
by
Patrick D. Schloss

Department of Microbiology & Immunology

University of Michigan
http://www.mothur.org

When using, please cite:
Schloss, P.D., et al., Introducing mothur: Open-source, platform-independent, community-supported software for describing and comparing microbial communities. Appl Environ Microbiol, 2009. 75(23):7537-41.

Distributed under the GNU General Public License

Type 'help()' for information on the commands that are available

For questions and analysis support, please visit our forum at https://forum.mothur.org

Type 'quit()' to exit program

[NOTE]: Setting random seed to 19760620.

Interactive Mode



mothur > summary.single(shared=trimmed.agc.shared, groupmode=t, calc=sobs-shannon-chao-invsimpson)

Processing group TARA_064_MES_IO

0.03

Processing group TARA_065_SRF_IO



## Rarefaction
```bash
mothur
>rarefaction.single(shared=trimmed.agc.shared)
```

In [16]:

remove("trimmed.agc.groups.rarefaction")
run_mothur(
    "rarefaction.single(shared=trimmed.agc.shared)",
    "11_mothur_rarefaction.log"
)


$ printf "rarefaction.single(shared=trimmed.agc.shared)\nquit()\n" | mothur
Linux version

Using ReadLine
mothur v.1.48.0
Last updated: 5/20/22
by
Patrick D. Schloss

Department of Microbiology & Immunology

University of Michigan
http://www.mothur.org

When using, please cite:
Schloss, P.D., et al., Introducing mothur: Open-source, platform-independent, community-supported software for describing and comparing microbial communities. Appl Environ Microbiol, 2009. 75(23):7537-41.

Distributed under the GNU General Public License

Type 'help()' for information on the commands that are available

For questions and analysis support, please visit our forum at https://forum.mothur.org

Type 'quit()' to exit program

[NOTE]: Setting random seed to 19760620.

Interactive Mode



mothur > rarefaction.single(shared=trimmed.agc.shared)

Using 16 processors.

Processing group TARA_064_MES_IO

0.03

Processing group TARA_065_SRF_IO

0.03

Processing group TARA_152_MES_NAO

0.03

Processing group TA

## Beta-diversity
```bash
mothur
>summary.shared(shared=trimmed.agc.shared, calc=sharedsobs-sorclass-jabund)
```

In [17]:

remove("trimmed.agc.summary")
run_mothur(
    "summary.shared(shared=trimmed.agc.shared, calc=sharedsobs-sorclass-jabund)",
    "12_mothur_beta.log"
)


$ printf "summary.shared(shared=trimmed.agc.shared, calc=sharedsobs-sorclass-jabund)\nquit()\n" | mothur
Linux version

Using ReadLine
mothur v.1.48.0
Last updated: 5/20/22
by
Patrick D. Schloss

Department of Microbiology & Immunology

University of Michigan
http://www.mothur.org

When using, please cite:
Schloss, P.D., et al., Introducing mothur: Open-source, platform-independent, community-supported software for describing and comparing microbial communities. Appl Environ Microbiol, 2009. 75(23):7537-41.

Distributed under the GNU General Public License

Type 'help()' for information on the commands that are available

For questions and analysis support, please visit our forum at https://forum.mothur.org

Type 'quit()' to exit program

[NOTE]: Setting random seed to 19760620.

Interactive Mode



mothur > summary.shared(shared=trimmed.agc.shared, calc=sharedsobs-sorclass-jabund)

Using 16 processors.
0.03 

Output File Names:
trimmed.agc.summary


mothur > quit()
Logfile : mothur.17

## Venn diagram
```bash
mothur
>venn(shared=trimmed.agc.shared)
```

In [18]:

remove_glob("trimmed.agc.0.03.sharedsobs*")
run_mothur(
    "venn(shared=trimmed.agc.shared)",
    "13_mothur_venn.log"
)


$ printf "venn(shared=trimmed.agc.shared)\nquit()\n" | mothur
Linux version

Using ReadLine
mothur v.1.48.0
Last updated: 5/20/22
by
Patrick D. Schloss

Department of Microbiology & Immunology

University of Michigan
http://www.mothur.org

When using, please cite:
Schloss, P.D., et al., Introducing mothur: Open-source, platform-independent, community-supported software for describing and comparing microbial communities. Appl Environ Microbiol, 2009. 75(23):7537-41.

Distributed under the GNU General Public License

Type 'help()' for information on the commands that are available

For questions and analysis support, please visit our forum at https://forum.mothur.org

Type 'quit()' to exit program

[NOTE]: Setting random seed to 19760620.

Interactive Mode



mothur > venn(shared=trimmed.agc.shared)
0.03

Output File Names: 
trimmed.agc.0.03.sharedsobs.TARA_064_MES_IO-TARA_065_SRF_IO-TARA_152_MES_NAO-TARA_152_SRF_NAO.svg
trimmed.agc.0.03.sharedsobs.TARA_064_MES_IO-TARA_065_SRF_IO-TARA_15

## Sample similarity tree
```bash
mothur
>tree.shared(shared=trimmed.agc.shared, calc=jabund)
```

In [19]:

remove("trimmed.agc.jabund.0.03.tre")
run_mothur(
    "tree.shared(shared=trimmed.agc.shared, calc=jabund)",
    "14_mothur_tree.log"
)


$ printf "tree.shared(shared=trimmed.agc.shared, calc=jabund)\nquit()\n" | mothur
Linux version

Using ReadLine
mothur v.1.48.0
Last updated: 5/20/22
by
Patrick D. Schloss

Department of Microbiology & Immunology

University of Michigan
http://www.mothur.org

When using, please cite:
Schloss, P.D., et al., Introducing mothur: Open-source, platform-independent, community-supported software for describing and comparing microbial communities. Appl Environ Microbiol, 2009. 75(23):7537-41.

Distributed under the GNU General Public License

Type 'help()' for information on the commands that are available

For questions and analysis support, please visit our forum at https://forum.mothur.org

Type 'quit()' to exit program

[NOTE]: Setting random seed to 19760620.

Interactive Mode



mothur > tree.shared(shared=trimmed.agc.shared, calc=jabund)

Using 16 processors.
0.03 
1
1

Output File Names: 
trimmed.agc.jabund.0.03.tre


mothur > quit()
Logfile : mothur.1790928007.logfile



# IV. Phylogenetic analysis

## Representative OTU sequences
```bash
python3 helper.py otu-rep trimmed.agc.list trimmed.fa
```

In [20]:

remove("oturep.fa")
run(
    "python3 helper.py otu-rep trimmed.agc.list trimmed.fa",
    "15_otu_rep.log"
)


$ python3 helper.py otu-rep trimmed.agc.list trimmed.fa
Representative sequences written at oturep.fa


## Build SILVA BLAST database
```bash
makeblastdb -in silva_ssu_db.fasta -dbtype nucl -out silva_blastdb
```

In [21]:

remove_glob("silva_blastdb*")
run(
    "makeblastdb -in silva_ssu_db.fasta -dbtype nucl -out silva_blastdb",
    "16_makeblastdb.log"
)


$ makeblastdb -in silva_ssu_db.fasta -dbtype nucl -out silva_blastdb


Building a new DB, current time: 10/02/2026 10:00:18
New DB name:   /home/gman/Documents/MS-GENIOMHE/M2-GENIOMHE/3. Advanced Genomics/3. Metagenomics/automated/results/work/silva_blastdb
New DB title:  silva_ssu_db.fasta
Sequence type: Nucleotide
Keep MBits: T
Maximum file size: 1000000000B
Adding sequences from FASTA; added 191162 sequences in 3.8533 seconds.




## BLAST against SILVA

```bash
blastn -query oturep.fa \
-db silva_blastdb \
-out oturep_silva_aln.blastn \
-evalue 0.001 \
-perc_identity 97 \
-outfmt "6 qseqid sseqid evalue pident qcovs" \
-max_target_seqs 1 \
-num_threads 4
```

In [22]:

remove("oturep_silva_aln.blastn")
run(
    'blastn -query oturep.fa '
    '-db silva_blastdb '
    '-out oturep_silva_aln.blastn '
    '-evalue 0.001 -perc_identity 97 '
    '-outfmt "6 qseqid sseqid evalue pident qcovs" '
    '-max_target_seqs 1 -num_threads 4',
    "17_blastn.log"
)


$ blastn -query oturep.fa -db silva_blastdb -out oturep_silva_aln.blastn -evalue 0.001 -perc_identity 97 -outfmt "6 qseqid sseqid evalue pident qcovs" -max_target_seqs 1 -num_threads 4


### Question 12

In [23]:

total = sum(1 for line in open(WORK / "oturep.fa") if line.startswith(">"))
matched = sum(1 for line in open(WORK / "oturep_silva_aln.blastn") if line.strip())
print("Total OTU representatives:", total)
print("Matched in SILVA:", matched)
print("Not matched:", total - matched)


Total OTU representatives: 21716
Matched in SILVA: 6220
Not matched: 15496


## Join taxonomy
```bash
python3 helper.py join-taxonomy oturep_silva_aln.blastn silva_ssu_db_taxa.txt
```

In [24]:

remove("oturep_tax_ids.txt", "oturep_tax_abundance.txt")
run(
    "python3 helper.py join-taxonomy oturep_silva_aln.blastn silva_ssu_db_taxa.txt",
    "18_join_taxonomy.log"
)


$ python3 helper.py join-taxonomy oturep_silva_aln.blastn silva_ssu_db_taxa.txt


## Build taxonomy tree with ETE3

TP command:

```bash
ete3 ncbiquery --tree --search [paste here] > oturep.tre
```

The cell below only automates the paste.


In [25]:

remove("oturep.tre")
tax_ids = " ".join((WORK / "oturep_tax_ids.txt").read_text().split())

run(
    f"ete3 ncbiquery --tree --search {tax_ids} > oturep.tre",
    "19_ete3_tree.log"
)


$ ete3 ncbiquery --tree --search 56765 77133 203449 1977864 256318 166587 1457863 526644 91374 759664 213663 1797217 527640 155900 407035 100272 86473 327939 415540 204529 115547 160660 171953 499814 152507 655815 187379 239730 69368 114243 652676 408172 344747 45670 164851 179113 120965 163362 100234 155098 1883416 526650 9606 4932 278095 1549806 203437 1660251 1736352 526693 1348774 1215650 1908525 526645 215611 1137138 69666 932002 699266 329652 257762 1096243 8030 153809 280754 1884369 187406 13136 1461752 34034 1701 173971 261514 648178 182957 161758 1889229 49265 221724 91750 651141 1898104 314107 360837 554078 313618 72000 1653778 165436 1041365 282483 526639 1905359 344907 547051 142194 1002672 152704 115413 512008 214040 187300 1331060 723580 1158509 1891201 813 168769 341161 573 526665 1761453 1300258 274854 233181 29575 198431 1561972 988 526657 337176 158080 1652797 368195 1456245 1658295 526643 1729581 1936125 195045 175278 526642 1810504 1925336 191625 398720 1123279 9330

# Outputs

Analysis files are in:

```text
results/work/
```

Terminal output logs are in:

```text
results/logs/
```
